# Load dataset

In [26]:
import h5py

def get_dataset_name(file_name_with_dir):
    filename_without_dir = file_name_with_dir.split('/')[-1]
    temp = filename_without_dir.split('_')[:-1]
    dataset_name = "_".join(temp)
    return dataset_name

filename_path = "Final Project data/Intra/train/rest_105923_1.h5"
with h5py.File(filename_path, 'r') as f:
    dataset_name = get_dataset_name(filename_path)
    matrix = f.get(dataset_name)[()]
    print(type(matrix))
    print(matrix.shape)

<class 'numpy.ndarray'>
(248, 35624)


In [27]:
import os
import numpy as np

CLASSES = ["rest", "task_motor", "task_story_math", "task_working_memory"]

def load_split(folder):
    X, y = [], []
    for file_name in sorted(os.listdir(folder)):
        if not file_name.endswith(".h5"):
            continue
        file_name_with_dir = os.path.join(folder, file_name)
        with h5py.File(file_name_with_dir, 'r') as f:
            dataset_name = get_dataset_name(file_name_with_dir)
            matrix = f.get(dataset_name)[()]
        X.append(matrix)
        y.append(next(i for i, c in enumerate(CLASSES) if file_name.startswith(c)))
    return np.array(X), np.array(y)

X_intra_train, y_intra_train = load_split("Final Project data/Intra/train")
X_intra_test,  y_intra_test  = load_split("Final Project data/Intra/test")

X_cross_train, y_cross_train = load_split("Final Project data/Cross/train")
X_cross_test1, y_cross_test1 = load_split("Final Project data/Cross/test1")
X_cross_test2, y_cross_test2 = load_split("Final Project data/Cross/test2")
X_cross_test3, y_cross_test3 = load_split("Final Project data/Cross/test3")

# Downsampling
Most MEG/EEG research uses 250 HZ. For this reason N=8 (~254HZ) Cite: https://pmc.ncbi.nlm.nih.gov/articles/PMC11372824/, https://mne.tools/stable/auto_tutorials/preprocessing/30_filtering_resampling.html

In [28]:
N = 8

splits = [X_intra_train, X_intra_test, X_cross_train, X_cross_test1, X_cross_test2, X_cross_test3]
downsampled_splits = [X[:, :, ::N] for X in splits]

# Data Preprocess 
Normalise time-wise, each time step independently across the 248 sensors using z-score. The reason for z-score standardization is because (1) it centers the data around 0, (2) its commonly used in MEG/EEG analysis and (3) it is less sensitive to extreme values in the signal compared to for example min-max scaling. cite: https://www.sciencedirect.com/science/article/pii/S0952197623003895



In [29]:
import numpy as np

def z_score_normalization(X_dataset):
    mean = np.mean(X_dataset, axis=2, keepdims=True)
    std = np.std(X_dataset, axis=2, keepdims=True)
    
    X_normalised = (X_dataset - mean) / std
    
    return X_normalised

normalised_splits = [z_score_normalization(X) for X in downsampled_splits]

X_intra_train_normalised, X_intra_test_normalised, X_cross_train_normalised, X_cross_test1_normalised, X_cross_test2_normalised, X_cross_test3_normalised = normalised_splits

# Save preprocessed datasets

In [30]:
def save_dataset(dataset_name, dataset):
    np.save(file=dataset_name, arr=dataset)

save_dataset("Preprocessed data/X_intra_train", X_intra_train_normalised)
save_dataset("Preprocessed data/X_intra_test",  X_intra_test_normalised)
save_dataset("Preprocessed data/X_cross_train", X_cross_train_normalised)
save_dataset("Preprocessed data/X_cross_test1", X_cross_test1_normalised)
save_dataset("Preprocessed data/X_cross_test2", X_cross_test2_normalised)
save_dataset("Preprocessed data/X_cross_test3", X_cross_test3_normalised)

save_dataset("Preprocessed data/y_intra_train", y_intra_train)
save_dataset("Preprocessed data/y_intra_test",  y_intra_test)
save_dataset("Preprocessed data/y_cross_train", y_cross_train)
save_dataset("Preprocessed data/y_cross_test1", y_cross_test1)
save_dataset("Preprocessed data/y_cross_test2", y_cross_test2)
save_dataset("Preprocessed data/y_cross_test3", y_cross_test3)